In [1]:
!pip install pandas

In [2]:
import pandas as pd
from pathlib import Path

data_dir = Path("UCI ML Drug Review dataset")

train_df = pd.read_csv(data_dir / "drugsComTrain_raw.csv")
test_df = pd.read_csv(data_dir / "drugsComTest_raw.csv")

print(train_df.shape)
print(test_df.shape)

(161297, 7)
(53766, 7)


In [3]:
train_df.info()
train_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 161297 entries, 0 to 161296
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   uniqueID     161297 non-null  int64
 1   drugName     161297 non-null  str  
 2   condition    160398 non-null  str  
 3   review       161297 non-null  str  
 4   rating       161297 non-null  int64
 5   date         161297 non-null  str  
 6   usefulCount  161297 non-null  int64
dtypes: int64(3), str(4)
memory usage: 8.6 MB


,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,3-Nov-15,10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,27-Nov-16,37


In [4]:
train_df.isna().sum()

uniqueID         0
drugName         0
condition      899
review           0
rating           0
date             0
usefulCount      0
dtype: int64

In [5]:
train_df['condition'] = train_df['condition'].fillna('Unknown')
test_df['condition'] = test_df['condition'].fillna('Unknown')

In [6]:
train_df['condition'].value_counts().tail(20)

condition
Cyclitis                                        1
Upper Limb Spasticity                           1
95</span> users found this comment helpful.     1
61</span> users found this comment helpful.     1
Diagnostic Bronchograms                         1
Neoplastic Diseases                             1
51</span> users found this comment helpful.     1
Mycoplasma Pneumonia                            1
Linear IgA Disease                              1
Subarachnoid Hemorrhage                         1
Myeloproliferative Disorders                    1
ungal Pneumonia                                 1
145</span> users found this comment helpful.    1
Scleroderma                                     1
Zollinger-Ellison Syndrome                      1
Tinea Barbae                                    1
Acute Nonlymphocytic Leukemia                   1
62</span> users found this comment helpful.     1
92</span> users found this comment helpful.     1
Neutropenia                             

In [7]:
import re

# pattern matches things like "95</span> users found this comment helpful."
garbage_pattern = r'</span>\s*users found this comment helpful'

garbage_mask_train = train_df['condition'].str.contains(garbage_pattern, regex=True, na=False)
garbage_mask_test = test_df['condition'].str.contains(garbage_pattern, regex=True, na=False)

print(f"Garbage condition rows in train: {garbage_mask_train.sum()}")
print(f"Garbage condition rows in test: {garbage_mask_test.sum()}")

train_df.loc[garbage_mask_train, 'condition'] = 'Unknown'
test_df.loc[garbage_mask_test, 'condition'] = 'Unknown'

# now handle the true NaNs same as before
train_df['condition'] = train_df['condition'].fillna('Unknown')
test_df['condition'] = test_df['condition'].fillna('Unknown')

Garbage condition rows in train: 900
Garbage condition rows in test: 271


In [8]:
train_df['condition'].value_counts().tail(20)

condition
Gastric Cance                    1
ibrocystic Breast Disease        1
ungal Infection Prophylaxis      1
Short Stature                    1
Hypercalcemia                    1
Coccidioidomycosis               1
Cyclitis                         1
Upper Limb Spasticity            1
Diagnostic Bronchograms          1
Neoplastic Diseases              1
Mycoplasma Pneumonia             1
Linear IgA Disease               1
Subarachnoid Hemorrhage          1
Myeloproliferative Disorders     1
ungal Pneumonia                  1
Scleroderma                      1
Zollinger-Ellison Syndrome       1
Tinea Barbae                     1
Acute Nonlymphocytic Leukemia    1
Neutropenia                      1
Name: count, dtype: int64

In [10]:
import html

# Unescape HTML entities (&#039;, &quot;, &amp;, etc.)
train_df['review'] = train_df['review'].apply(html.unescape)
test_df['review'] = test_df['review'].apply(html.unescape)

# Strip leading/trailing stray quote characters left over from the original file's formatting
train_df['review'] = train_df['review'].str.strip('"')
test_df['review'] = test_df['review'].str.strip('"')

# Sanity check — should be ~0 after unescaping
print(train_df['review'].str.contains(r'&#\d+;|&quot;|&amp;', regex=True).sum())

0


In [11]:
train_df.assign(review_len=train_df['review'].str.len()) \
    .sort_values('review_len') \
    .head(20)[['drugName', 'condition', 'review', 'rating', 'review_len']]

,drugName,condition,review,rating,review_len
113792,Keppra,Neuralgia,I,9,1
42909,Levetiracetam,Neuralgia,I,9,1
93044,Linagliptin / metformin,min),G,9,1
128652,Ifex,Testicular Cance,-,10,1
123174,Ifosfamide,Testicular Cance,-,10,1
143619,Clonazepam,Anxiety,I,8,1
148114,Tramadol,Back Pain,Ok,6,2
30621,Levothyroxine,Not Listed / Othe,hi,10,2
95222,Tadalafil,Erectile Dysfunction,Ok,6,2
128311,Nucynta,Pain,ok,6,2


## Data Cleaning — Summary So Far

We loaded the train (161,297 rows) and test (53,766 rows) sets and confirmed both came in fully intact with no parsing errors, matching the dataset's expected size. All columns were complete except `condition`, which had 899 missing values in train. On inspection, we also found that `condition` contained non-medical garbage strings caused by a scraping artifact — HTML helpfulness-count snippets (e.g. `"95</span> users found this comment helpful."`) had leaked into the field instead of being filtered out during the original scrape. We identified 900 such rows in train and 271 in test, and combined them with the true missing values, imputing all as `"Unknown"` rather than dropping the rows.

We then cleaned the `review` text by unescaping HTML entities (`&#039;`, `&quot;`, `&amp;`, etc.) and stripping stray leading/trailing quote characters left over from the original file's formatting. While inspecting the shortest reviews, we noticed a small number reduced to a single character (e.g. `"I"`, `"-"`, `"G"`) — these carry essentially no usable information for predicting rating. We chose to leave them in the dataset rather than drop them, noting them as a known source of low-quality signal to keep in mind when interpreting model performance.


In [12]:
import os

os.makedirs('data', exist_ok=True)

train_df.to_csv('data/train_clean.csv', index=False)
test_df.to_csv('data/test_clean.csv', index=False)

print("Saved:")
print("train_clean.csv:", train_df.shape)
print("test_clean.csv:", test_df.shape)

Saved:
train_clean.csv: (161297, 7)
test_clean.csv: (53766, 7)
